In [5]:
import os
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from torch.utils.data import TensorDataset, DataLoader
from torch.serialization import safe_globals

In [ ]:
# ---------------------------------------
# Configuration
# ---------------------------------------
parquet_path   = 'Master_cleaned.parquet'
return_col     = 'ret_5d_future'
batch_size     = 1_000_000   # streaming size for Parquet
test_fraction  = 0.2         # last 20% of dates for test
device         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# ---------------------------------------
# Build X_test, y_test (same as before)
# ---------------------------------------
pf = pq.ParquetFile(parquet_path)
all_cols = pf.schema_arrow.names
exclude  = {return_col, 'date', 'ticker'}
feature_cols = [c for c in all_cols if c not in exclude]

cutoff_lists, all_dates = {}, set()
for batch in pf.iter_batches(batch_size=batch_size):
    dfb = batch.to_pandas()[['date', return_col]]
    for date, grp in dfb.groupby('date'):
        all_dates.add(date)
        cutoff_lists.setdefault(date, []).extend(grp[return_col].values)
cutoff_map = {d: np.quantile(vals, 0.95) for d, vals in cutoff_lists.items()}
all_dates = sorted(all_dates)

split_idx  = int(len(all_dates) * (1 - test_fraction))
test_dates = set(all_dates[split_idx:])
# split_idx = next(i for i, d in enumerate(all_dates) if str(d) >= '2025-06-15')
# test_dates = set(all_dates[split_idx:])

sample = pf.read_row_group(0).to_pandas()[feature_cols]
scaler = StandardScaler().fit(sample.values)

chunks = []
for batch in pf.iter_batches(batch_size=batch_size):
    df = batch.to_pandas()
    df = df[df['date'].isin(test_dates)]
    if not df.empty:
        chunks.append(df)
test_df = pd.concat(chunks, ignore_index=True)

test_df['label'] = (test_df[return_col] >= test_df['date'].map(cutoff_map)).astype(np.int8)
mask = np.isfinite(test_df[feature_cols].values).all(axis=1)
test_df = test_df.loc[mask].reset_index(drop=True)

X_test = scaler.transform(test_df[feature_cols].values.astype(np.float32))
y_test = test_df['label'].values.astype(np.int8)
print("X_test shape:", X_test.shape, "y_test dist:", np.bincount(y_test))

# ---------------------------------------
# Load the TorchScript model
# ---------------------------------------
scripted_model = torch.jit.load('selector_mlp_scripted.pt', map_location=device)
scripted_model.to(device).eval()

# ---------------------------------------
# Batched inference & metrics
# ---------------------------------------
test_ds     = TensorDataset(torch.from_numpy(X_test).float(),
                            torch.from_numpy(y_test).long())
test_loader = DataLoader(test_ds, batch_size=2048, shuffle=False, num_workers=0)

all_probs, all_preds, all_labels = [], [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        Xb = Xb.to(device)
        logits = scripted_model(Xb)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(np.int8)

        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(yb.numpy())

all_probs  = np.concatenate(all_probs)
all_preds  = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

auc      = roc_auc_score(all_labels, all_probs)
accuracy = accuracy_score(all_labels, all_preds)

print(f"\nTest AUC:      {auc:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n",
      classification_report(all_labels, all_preds, digits=4))

test_df["prob"] = all_probs
test_df["pred"] = all_preds  # Optional: for inspection
test_df["rank_prob"] = test_df.groupby("date")["prob"].rank(method="first", ascending=False)
top_150_df = test_df[test_df["rank_prob"] <= 150].copy()
top_150_df = top_150_df.sort_values(["date", "rank_prob"])
top_150_df[["date", "ticker", "prob", "rank_prob", "ret_5d_future", "open"]].to_csv("top_150_predictions.csv", index=False)
print("Exported top 150 predictions per date to top_150_predictions.csv")

Using device: cuda
X_test shape: (19745254, 37) y_test dist: [18706030  1039224]

Test AUC:      0.8066
Test Accuracy: 0.6832

Classification Report:
               precision    recall  f1-score   support

           0     0.9846    0.6762    0.8017  18706030
           1     0.1219    0.8091    0.2119   1039224

    accuracy                         0.6832  19745254
   macro avg     0.5532    0.7427    0.5068  19745254
weighted avg     0.9392    0.6832    0.7707  19745254

Exported top 150 predictions per date to top_150_predictions.csv


In [8]:
selected = test_df[test_df['pred'] == 1]
print(len(selected[['date', 'ticker', 'prob']]))
selected = selected.sort_values(by='prob', ascending=False)
selected

6898570


,date,ticker,open,high,low,close,volume,dow_0,dow_1,dow_2,...,neutral_score,pos_score,pos_minus_neg,pos_minus_neg_lag1,headline_count,headline_count_lag1,label,prob,pred,rank_prob
4941712,2017-07-03 00:00:00+00:00,TOPS,2.736000e+07,2.808000e+07,2.304000e+07,2.664000e+07,0.025000,1,0,0,...,0.0,0.0,0.0,-0.335662,0.0,1.0,0,1.000000,1,1.0
4949465,2017-07-06 00:00:00+00:00,TOPS,2.448000e+07,2.556000e+07,2.160000e+07,2.288160e+07,0.029167,0,0,0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0,1.000000,1,1.0
4956346,2017-07-07 00:00:00+00:00,TOPS,2.232000e+07,2.303280e+07,1.908000e+07,1.946880e+07,0.025000,0,0,0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0,1.000000,1,1.0
4942327,2017-07-05 00:00:00+00:00,TOPS,2.981520e+07,3.276000e+07,2.448000e+07,2.591280e+07,0.083333,0,0,1,...,0.0,0.0,0.0,0.000000,0.0,0.0,0,1.000000,1,1.0
4959452,2017-07-10 00:00:00+00:00,TOPS,2.016000e+07,2.036880e+07,1.800000e+07,1.864800e+07,0.012500,1,0,0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0,1.000000,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5267088,2017-10-06 00:00:00+00:00,SKF,2.629680e+02,2.662680e+02,2.606910e+02,2.645660e+02,3251.243015,0,0,0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0,0.500001,1,1799.0
3074355,2015-11-16 00:00:00+00:00,SIJ,2.858180e+02,2.858180e+02,2.798140e+02,2.798140e+02,736.211000,1,0,0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0,0.500000,1,1479.0
10817412,2021-07-12 00:00:00+00:00,PLTM,1.094000e+01,1.107000e+01,1.084200e+01,1.103000e+01,22889.000000,1,0,0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0,0.500000,1,2471.0
8681706,2020-03-30 00:00:00+00:00,AHCO,1.555000e+01,1.575000e+01,1.508000e+01,1.572000e+01,59990.000000,1,0,0,...,0.0,0.0,0.0,0.000000,0.0,0.0,1,0.500000,1,3100.0


In [10]:
top_150_df[["date", "ticker", "prob", "rank_prob", "ret_5d_future", "open"]].to_csv("top_150_predictions.csv", index=False)